# 🏥 High-Accuracy Kidney Disease AI Model Training (VGG16 -> Single-File ONNX)
### Trains on 12,446 CT Scan images (`Cyst`, `Normal`, `Stone`, `Tumor`) in ~3 minutes on free GPU.

In [ ]:
# Step 1: Install Exporter Dependencies
!pip install -q onnx onnxscript

import os, time, copy, zipfile, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import onnx

# Check GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Training Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Step 2: Locate Dataset (Universal: Kaggle Notebook, Uploaded Zip, or Drive)
os.makedirs('./dataset', exist_ok=True)

if os.path.exists('/kaggle/input'):
    print("⚡ Detected Kaggle Notebook environment! Dataset is already mounted at /kaggle/input.")
    search_dir = '/kaggle/input'
elif os.path.exists('archive.zip'):
    print("📦 Found uploaded archive.zip! Unzipping into ./dataset...")
    with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
        zip_ref.extractall('./dataset')
    search_dir = './dataset'
else:
    search_dir = './dataset'

# Auto-detect folder containing Cyst, Normal, Stone, Tumor
target_classes = {'cyst', 'normal', 'stone', 'tumor'}
DATASET_ROOT = None
for root, dirs, files_list in os.walk(search_dir):
    lower_dirs = {d.lower() for d in dirs}
    if target_classes.issubset(lower_dirs):
        DATASET_ROOT = root
        break

print(f"🎯 Dataset root verified at: {DATASET_ROOT}")
assert DATASET_ROOT is not None, "Could not locate Cyst/Normal/Stone/Tumor subfolders!"

In [ ]:
# Step 3: Fast DataLoader Setup (num_workers=0 to prevent Kaggle deadlock)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(DATASET_ROOT, transform=transform_train)
class_names = full_dataset.classes
print(f"📊 Total Images: {len(full_dataset)} | Classes ({len(class_names)}): {class_names}")

train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

# Note: num_workers=0 is REQUIRED on Kaggle to avoid IPC shared memory hang
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False, num_workers=0)
print(f"✅ Ready: {len(train_loader)} batches per epoch.")

In [ ]:
# Step 4: Build VGG16 (Pretrained or Offline Fallback)
print("🧠 Initializing VGG16 Architecture...")
try:
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    print("✅ Loaded Pretrained ImageNet Weights!")
except Exception as e:
    print("⚠️ No Internet detected: Running in 100% Offline Mode!")
    model = models.vgg16(weights=None)
    for param in model.features.parameters():
        param.requires_grad = True

# Classifier head optimized for kidney diagnosis
model.classifier = nn.Sequential(
    nn.Linear(25088, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.4),
    nn.Linear(256, 64),
    nn.ReLU(inplace=True),
    nn.Dropout(p=0.2),
    nn.Linear(64, len(class_names))
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4, weight_decay=1e-3)
scaler = torch.cuda.amp.GradScaler()
print("✅ Model successfully prepared on GPU!")

In [ ]:
# Step 5: Fast GPU Training with Live Batch Progress
NUM_EPOCHS = 3
best_acc = 0.0
best_weights = copy.deepcopy(model.state_dict())

print("🚀 Starting Training Loop on GPU...")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct = 0.0, 0
    total_samples = 0
    print(f"\n--- Epoch [{epoch+1}/{NUM_EPOCHS}] ---")
    
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        _, preds = torch.max(outputs, 1)
        train_loss += loss.item() * inputs.size(0)
        train_correct += torch.sum(preds == labels.data)
        total_samples += labels.size(0)
        
        # Live progress print every 30 batches
        if (i + 1) % 30 == 0 or (i + 1) == len(train_loader):
            current_acc = (train_correct.double() / total_samples).item() * 100
            print(f"  ⚡ Batch [{i+1}/{len(train_loader)}] | Train Acc: {current_acc:.1f}% | Loss: {loss.item():.4f}")
        
    # Validation Phase
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            val_loss += loss.item() * inputs.size(0)
            val_correct += torch.sum(preds == labels.data)
            
    epoch_val_acc = (val_correct.double() / val_size).item()
    print(f"🎯 Epoch [{epoch+1}/{NUM_EPOCHS}] Finished -> Val Accuracy: {epoch_val_acc*100:.2f}%")
    
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        best_weights = copy.deepcopy(model.state_dict())

total_time = time.time() - start_time
print(f"\n🎉 Training Complete in {total_time/60:.1f} minutes! Best Validation Accuracy: {best_acc*100:.2f}%")
model.load_state_dict(best_weights)

In [ ]:
# Step 6: Single-File ONNX Export & Download
model.eval().to('cpu')

dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)
output_file = "model.onnx"

print("📦 Exporting to single self-contained ONNX file...")
torch.onnx.export(
    model,
    dummy_input,
    output_file,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Ensure weights are 100% bundled into one file (NO split .data files)
proto = onnx.load(output_file)
onnx.save_model(proto, output_file, save_as_external_data=False)
size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"✅ Exported Single-File ONNX! File size: {size_mb:.2f} MB")

# Handle download across Google Colab & Kaggle Notebooks
try:
    from google.colab import files
    print("📥 Downloading model.onnx directly to your computer...")
    files.download(output_file)
except ImportError:
    from IPython.display import FileLink, display
    print("📥 Download model.onnx by clicking below (or from the Output panel on the right):")
    display(FileLink(output_file))